# INP浓度的时间序列

在一张图上用不同颜色绘制各个温度的INP浓度(月均, 周均, 日均, 原始), errorbar用`median + IQR`表示

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.optimize import curve_fit

In [ ]:
df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.3.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

In [ ]:
def is_extreme(series):
    lower_bound = series.quantile(0.01)
    upper_bound = series.quantile(0.99)
    return (series < lower_bound) | (series > upper_bound)

In [ ]:
def is_outlier(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 3 * iqr
    upper_bound = q3 + 3 * iqr
    return (series < lower_bound) | (series > upper_bound)

## 月均INP浓度

In [ ]:
# 在一张图上用不同颜色绘制各个温度的INP浓度(月均), bar用IQR表示

TEMPERATURE = [-15, -20, -25, -30, -35]
colors = sns.color_palette("husl", len(TEMPERATURE))
fig, ax = plt.subplots(figsize=(8, 5))
for i, temp in enumerate(TEMPERATURE):
    # 数据质控
    df_temp = df[df['T_a(°C)'] == temp]

    conc_inp = df_temp['N_inp_net(#/L)']
    conc_inp_month = conc_inp.resample('ME').mean()
    conc_inp_month_median = conc_inp.resample('ME').median()
    q1 = conc_inp.resample('ME').quantile(0.25)
    q3 = conc_inp.resample('ME').quantile(0.75)
    lower_error = conc_inp_month_median - q1
    upper_error = q3 - conc_inp_month_median

    ax.errorbar(conc_inp_month_median.index, conc_inp_month_median.values,
                yerr=[lower_error, upper_error], fmt='o', color=colors[i], capsize=5, label=f'median T_a={temp}°C', zorder=10)# 使用1/4分位数和3/4分位数作为标准差

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m/%d"))
fig.autofmt_xdate()   # 自动旋转日期标签

ax.set_ylabel('conc of INP (#/L)')
ax.set_yscale('log')

ax.set_title('Monthly INP Concentration')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=6)

In [ ]:
# 月均INP浓度, errorbar分别用`mean + std`和`median + IQR`表示, 对比差异

TEMPERATURE = [-15, -20, -25, -30, -35]
colors = sns.color_palette("husl", len(TEMPERATURE))
fig, ax = plt.subplots(figsize=(8, 5))
for i, temp in enumerate(TEMPERATURE):
    df_temp = df[df['T_a(°C)'] == temp]

    conc_inp = df_temp['N_inp_net(#/L)']
    conc_inp_month = conc_inp.resample('ME').mean()
    conc_inp_month_median = conc_inp.resample('ME').median()
    q1 = conc_inp.resample('ME').quantile(0.25)
    q3 = conc_inp.resample('ME').quantile(0.75)
    lower_error = conc_inp_month_median - q1
    upper_error = q3 - conc_inp_month_median

    if temp == -30:
        ax.errorbar(conc_inp_month_median.index, conc_inp_month_median.values,
                yerr=[lower_error, upper_error], fmt='o', color=colors[i], capsize=5, label=f'median T_a={temp}°C', zorder=10)# 使用1/4分位数和3/4分位数作为标准差
        ax.errorbar(conc_inp_month.index, conc_inp_month.values,
                yerr=conc_inp_month.std(), fmt='o', color=colors[i+1], capsize=5, label=f'mean T_a={temp}°C')
    
    
#ax.xaxis.set_major_locator(mdates.DayLocator(interval=15))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m/%d"))
fig.autofmt_xdate()   # 自动旋转日期标签

ax.set_ylabel('conc of INP (#/L)')
#ax.set_yscale('log')

ax.set_title('Monthly INP Concentration')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=6)

## 周均INP浓度

In [ ]:
# 在一张图上用不同颜色绘制各个温度的INP浓度(周均), bar用IQR表示

TEMPERATURE = [-15, -20, -25, -30, -35]
colors = sns.color_palette("husl", len(TEMPERATURE))
fig, ax = plt.subplots(figsize=(12, 4))
for i, temp in enumerate(TEMPERATURE):
    df_temp = df[df['T_a(°C)'] == temp]

    conc_inp = df_temp['N_inp_net(#/L)']
    conc_inp_week = conc_inp.resample('W').mean()
    conc_inp_week_median = conc_inp.resample('W').median()
    q1 = conc_inp.resample('W').quantile(0.25)
    q3 = conc_inp.resample('W').quantile(0.75)
    lower_error = conc_inp_week_median - q1
    upper_error = q3 - conc_inp_week_median

    ax.errorbar(conc_inp_week_median.index, conc_inp_week_median.values,
                yerr=[lower_error, upper_error], fmt='o', color=colors[i], capsize=5, label=f'T_a={temp}°C', zorder=10)# 使用1/4分位数和3/4分位数作为标准差

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m/%d"))
fig.autofmt_xdate()   # 自动旋转日期标签

ax.set_ylabel('conc of INP (#/L)')
ax.set_yscale('log')

ax.set_title('Weekly INP Concentration')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=6)

## 日均INP浓度

In [ ]:
# 在一张图上用不同颜色绘制各个温度的INP浓度(日均), 未画误差bar(考虑到1.一个点的数据量较少;2.图片的清晰性)

TEMPERATURE = [-15, -20, -25, -30, -35]
colors = sns.color_palette("husl", len(TEMPERATURE))
fig, ax = plt.subplots(figsize=(12, 4))
for i, temp in enumerate(TEMPERATURE):
    df_temp = df[df['T_a(°C)'] == temp]

    conc_inp = df_temp['N_inp_net(#/L)']
    conc_inp_day = conc_inp.resample('D').mean()
    conc_inp_day_median = conc_inp.resample('D').median()
    q1 = conc_inp.resample('D').quantile(0.25)
    q3 = conc_inp.resample('D').quantile(0.75)
    lower_error = conc_inp_day_median - q1
    upper_error = q3 - conc_inp_day_median

    ax.plot(conc_inp_day.index, conc_inp_day.values, color=colors[i], label=f'{temp}°C') # 折线图
    ax.scatter(conc_inp_day.index, conc_inp_day.values, color=colors[i], s=10,label=f'{temp}°C', linewidth=0.6) # 散点图

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m/%d"))
fig.autofmt_xdate()   # 自动旋转日期标签

ax.set_ylabel('conc of INP (#/L)')
ax.set_yscale('log')

ax.set_title('Daily INP Concentration')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=6)

## 原始INP浓度(次日级)

In [ ]:
# 在一张图上用不同颜色绘制各个温度的INP浓度(原始)

TEMPERATURE = [-15, -20, -25, -30, -35]
colors = sns.color_palette("husl", len(TEMPERATURE))
fig, ax = plt.subplots(figsize=(12, 4))
for i, temp in enumerate(TEMPERATURE):
    df_temp = df[df['T_a(°C)'] == temp]

    conc_inp = df_temp['N_inp_net(#/L)']
    ax.scatter(conc_inp.index, conc_inp.values, color=colors[i], s=10, alpha=0.5, label=f'temp={temp}°C', linewidth=0.6)


ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m/%d"))
fig.autofmt_xdate()   # 自动旋转日期标签

ax.set_ylabel('conc of INP (#/L)')
ax.set_yscale('log')

ax.set_title('Raw INP Concentration')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=6)

### fill_between 绘制置信区间

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.ticker as ticker

# ==========================================
# 0. 全局学术图表格式设置 (Thesis Standard)
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True

# ==========================================
# 1. 生成模拟数据 (请替换为你自己的 DataFrame)
# ==========================================
df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.csv")
temps = [-15, -20, -25, -30, -35]
df['Date'] = pd.to_datetime(df['Date'])
# ==========================================
# 2. 准备绘图参数
# ==========================================
# 颜色映射：为5个温度分配不同的颜色 (仿照原图的灰、红、蓝、绿等配色)
colors = {
    -15: '#A9A9A9', # 灰色
    -20: '#F08080', # 浅红色
    -25: '#4169E1', # 宝蓝色
    -30: '#9ACD32', # 黄绿色
    -35: '#DDA0DD'  # 梅红色 (新增一个颜色以凑齐5个)
}

# 创建画布，5行1列，共享X轴
fig, axs = plt.subplots(5, 1, figsize=(10, 12), sharex=True, dpi=500)
# 消除子图之间的垂直间距
fig.subplots_adjust(hspace=0)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, temp in enumerate(temps):
    ax = axs[i]
    
    # 提取当前温度的数据，并按时间排序 (计算滑动平均必须保证时间顺序)
    df_sub = df[df['T_a(°C)'] == temp].sort_values('Date').copy()
    
    # 计算10个散点的滑动平均 (若点数不足10个则计算现有的平均)
    df_sub['Moving_Avg'] = df_sub['N_inp_net(#/L)'].rolling(window=10, min_periods=1).mean()
    
    # --- A. 绘制置信区间 (粉色背景) ---
    # 假设 Significance_Level 是一条基准线，粉色区域在基准线以下
    ax.fill_between(df_sub['Date'], 1e-3, df_sub['Significance_Level(#/L)'], 
                    color='magenta', alpha=0.2, zorder=5)
    
    # --- B. 绘制原始散点 ---
    ax.scatter(df_sub['Date'], df_sub['N_inp_net(#/L)'], 
               s=60, alpha=0.9, c=colors[temp], edgecolor='black', linewidth=0.5, zorder=3)
    
    # --- C. 绘制滑动平均 (青色叉号) ---
    ax.scatter(df_sub['Date'], df_sub['Moving_Avg'], 
               s=50, marker='x', c='cyan', linewidth=1.5, zorder=4)
    
    # --- D. Y轴对数设置与刻度格式化 ---
    ax.set_yscale('log')
    ax.set_ylim(1e-3, 1e4) # 根据你的实际数据范围调整
    # 在ax的右侧显示y轴刻度
    if i % 2 == 1:  # 偶数行显示在右侧，奇数行显示在左侧
        ax.yaxis.tick_right()

    # 设置Y轴主刻度和次刻度，确保左右两侧都有向内的刻度
    ax.tick_params(axis='y', which='major', length=6, width=1.2, direction='in', left=True, right=True)
    ax.tick_params(axis='y', which='minor', length=3, width=1, direction='in', left=True, right=True)
    
    # --- E. 绘制图例 (重点：自定义仿原图格式) ---
    # 创建图例句柄
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[temp], 
               markeredgecolor='k', markersize=8, label=f'{temp} °C'),
        Patch(facecolor='magenta', alpha=0.2, edgecolor='none', label='Significance level'),
        Line2D([0], [0], marker='x', color='w', markeredgecolor='cyan', 
               markerfacecolor='cyan', markersize=8, label='Moving average (10 pts)')
    ]
    
    # 将图例分两列排布，类似原图紧凑放置在左上角或右下角
    # 参数 bbox_to_anchor 可用于微调图例的具体坐标位置
    ax.legend(handles=legend_elements, loc='upper right', ncol=2, frameon=True, 
              edgecolor='gray', fontsize=9, handletextpad=0.5, columnspacing=1)

# ==========================================
# 4. 整体格式调整 (X轴与全局标签)
# ==========================================
# X轴时间格式化：只显示年份
axs[-1].xaxis.set_major_locator(mdates.MonthLocator())
axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
# 设置X轴刻度样式，上下均有向内刻度
for ax in axs:
    ax.tick_params(axis='x', which='major', length=6, width=1.2, direction='in', bottom=True, top=True)

# 限制X轴的显示范围 (根据你的数据时间段调整)
axs[-1].set_xlim()

# 设置全局Y轴标签 (使用 fig.supylabel 可以保证标签居中于所有子图)
fig.supylabel('INP Concentration [L$^{-1}$]', x=0.04, fontsize=16)

# 留出边距并保存
plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.08)

#plt.savefig('INP_TimeSeries.png', dpi=500, bbox_inches='tight')
plt.show()

### 全部数据

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ==========================================
# 0. 全局学术图表格式设置
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True

# ==========================================
# 1. 读取数据
# ==========================================
df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.41.csv")
temps = [-15, -20, -25, -30, -35]
df['Date'] = pd.to_datetime(df['Date'])

# ==========================================
# 2. 准备绘图参数
# ==========================================
colors = {
    -15: '#FDB462', -20: '#F08080', -25: '#4169E1', 
    -30: '#9ACD32', -35: '#DDA0DD'
}
#
fig, axs = plt.subplots(5, 1, figsize=(10, 12), sharex=True, dpi=500)
fig.subplots_adjust(hspace=0)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, temp in enumerate(temps):
    ax = axs[i]
    
    # 提取并排序
    df_sub = df[df['T_a(°C)'] == temp].sort_values('Date').copy()
    
    moving_window = '7D'
    df_sub['Moving_Avg'] = df_sub.set_index('Date')['N_inp_net(#/L)'].rolling(moving_window).mean().values
    
    # --- A. 绘制置信区间 ---
    dt_mode = df_sub['Date'].diff().value_counts().index[0]
    bar_width = (dt_mode.total_seconds() / 86400) * 50
    ax.bar(df_sub['Date'], height=df_sub['Significance_Level(#/L)'], 
           width=bar_width, color='#A9A9A9', alpha=0.9, edgecolor='none', zorder=2)
    
    # --- B. 绘制原始散点 ---
    ax.scatter(df_sub['Date'], df_sub['N_inp_net(#/L)'], 
               s=50, alpha=0.8, c=colors[temp], edgecolor='black', linewidth=0.8, zorder=1)
    
    # --- C. 绘制滑动平均 ---
    # 【修改3】改为实线(Plot)或更细小的散点，避免喧宾夺主。这里演示实线形式：
    #ax.plot(df_sub['Date'], df_sub['Moving_Avg'], 
            #color='deepskyblue', linewidth=1.5, zorder=4, label='Moving average')
    
    # 如果你导师硬性要求必须用叉号，请用以下代码替换上面的 ax.plot：
    ax.scatter(df_sub['Date'], df_sub['Moving_Avg'], s=20, marker='x', c='#66C2A5', linewidth=1.0, zorder=3)
    
    # --- D. Y轴设置 ---
    ax.set_yscale('log')
    ax.set_ylim(1e-3, 1e4)
    
    if i % 2 == 1:
        ax.yaxis.tick_right()

    ax.tick_params(axis='y', which='major', length=6, width=1.2, direction='in', left=True, right=True)
    ax.tick_params(axis='y', which='minor', length=3, width=1, direction='in', left=True, right=True)
    
    # --- E. 绘制图例 ---
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[temp], 
               markeredgecolor='k', markersize=7, label=f'{temp} °C'),
        Patch(facecolor='#A9A9A9', alpha=0.4, edgecolor='none', label='Significance level'),
        # 如果上面选了实线，图例用这行：
        #Line2D([0], [0], color='deepskyblue', linewidth=1.5, label='Moving average')
        # 如果上面选了散点，图例用这行：
        Line2D([0], [0], marker='x', color='w', markeredgecolor='#66C2A5', markersize=6, linewidth=1.0, label=f'Moving average({moving_window})')
    ]
    
    ax.legend(handles=legend_elements, loc='upper right', ncol=3, frameon=True, 
              edgecolor='gray', fontsize=9, handletextpad=0.5, columnspacing=0.8)

# ==========================================
# 4. 整体格式调整
# ==========================================
axs[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=1)) # 每月显示一次
axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate(rotation=0, ha='center') # 防止X轴日期过长重叠，如果重叠可以把 rotation 改为 30
# 添加网格线 (灰色虚线，不抢眼)
for ax in axs:
    ax.grid(which='major', axis='y', linestyle='--', linewidth=0.5, color='gray', alpha=0.5)

for ax in axs:
    ax.tick_params(axis='x', which='major', length=6, width=1.2, direction='in', bottom=True, top=True)

fig.supylabel('INP Concentration [L$^{-1}$]', x=0.04, fontsize=16)
plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.08)

plt.show()

### 区分显著/不显著数据点

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ==========================================
# 0. 全局学术图表格式设置
# ==========================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
plt.rcParams['mathtext.fontset'] = 'custom'

# ==========================================
# 1. 读取数据
# ==========================================
df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv")

temps = [-15, -20, -25, -30, -35]
df['Time'] = pd.to_datetime(df['Time'])

# ==========================================
# 2. 准备绘图参数
# ==========================================
colors = {
    -15: '#FDB462', -20: '#F08080', -25: '#4169E1', 
    -30: '#9ACD32', -35: '#DDA0DD'
}
#
fig, axs = plt.subplots(5, 1, figsize=(10, 12), sharex=True, dpi=500)
fig.subplots_adjust(hspace=0)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, temp in enumerate(temps):
    ax = axs[i]
    
    # 提取并排序
    df_sig = df[(df['T_a(degC)'] == temp) & (df['Is_Significant'] == True)].sort_values('Time').copy()
    df_nonsig = df[(df['T_a(degC)'] == temp) & (df['Is_Significant'] == False)].sort_values('Time').copy()

    moving_window = '7D'
    df_sig['Moving_Avg'] = df_sig.set_index('Time')['N_INP(#/L)'].rolling(moving_window).mean().values

    # --- A. 绘制置信区间---
    if i == 0:
        dt_mode = df_sig['Time'].diff().value_counts().index[0]
    bar_width = (dt_mode.total_seconds() / 86400) * 50
    ax.bar(df_sig['Time'], height=df_sig['Significance_Level(#/L)'], 
           width=bar_width, color='#A9A9A9', alpha=0.9, edgecolor='none', zorder=2)
    
    # --- B. 绘制原始散点 ---
    ax.scatter(df_nonsig['Time'], df_nonsig['N_INP(#/L)'], 
               s=50, alpha=0.6, c='gray', edgecolor='none', zorder=0)
    ax.scatter(df_sig['Time'], df_sig['N_INP(#/L)'], 
               s=50, alpha=0.8, c=colors[temp], edgecolor='black', linewidth=0.8, zorder=1)
    
    
    # 使用叉号
    ax.scatter(df_sig['Time'], df_sig['Moving_Avg'], s=20, marker='x', c='#66C2A5', linewidth=1.0, zorder=3)

    # --- C. Y轴设置 ---
    ax.set_yscale('log')
    ax.set_ylim(10**-0.5, 10**3.5)
    
    if i % 2 == 1:
        ax.yaxis.tick_right()

    ax.tick_params(axis='y', which='major', length=6, width=1.2, direction='in', left=True, right=True)
    ax.tick_params(axis='y', which='minor', length=3, width=1, direction='in', left=True, right=True)
    
    # --- D. 绘制图例 ---
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[temp], 
               markeredgecolor='k', markersize=7, label=f'{temp} ± 1°C'),
        Patch(facecolor='#A9A9A9', alpha=0.4, edgecolor='none', label='Significance level'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=7, label='Not significant'),
        Line2D([0], [0], marker='x', color='w', markeredgecolor='#66C2A5', markersize=6, linewidth=1.0, label=f'Moving average(7-Days)')
    ]
    
    ax.legend(handles=legend_elements, loc='upper left', ncol=2, frameon=True, 
              edgecolor='gray', fontsize=9, handletextpad=0.5, columnspacing=0.8)

# ==========================================
# 4. 整体格式调整
# ==========================================
axs[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=1)) # 每月显示一次
axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate(rotation=0, ha='center') # 防止X轴日期过长重叠，如果重叠可以把 rotation 改为 30

# 添加网格线 (灰色虚线，不抢眼)
for ax in axs:
    ax.grid(which='major', axis='y', linestyle='--', linewidth=0.5, color='gray', alpha=0.5)

for ax in axs:
    ax.tick_params(axis='x', which='major', length=6, width=1.2, direction='in', bottom=True, top=True)

fig.supylabel('INP Concentration [L$^{-1}$]', x=0.04, fontsize=16)
plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.08)

plt.savefig(r'D:\Coding\master0_2025\Thesis\fig 2.png', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
df = pd.read_csv(r"E:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.41.csv")
N_inp = df['N_inp_net(#/L)']
# 输出不同温度下INP浓度的中位数, 算术平均数, 几何平均数
for temp in [-15, -20, -25, -30, -35]:
    print(f"Temperature: {temp}°C")
    N_temp = df['N_inp_net(#/L)'][df['T_a(°C)'] == temp]
    print("Median:", N_temp.median())
    print("Mean:", N_temp.mean())
    #print("Geometric Mean:", N_temp.prod() ** (1 / len(N_temp)))
    print(f"Significant data points: {df['Is_Significant'][df['T_a(°C)'] == temp].value_counts()[True]}")# 显著数据点的个数
    print(f"{df['Is_Significant'][df['T_a(°C)'] == temp].value_counts()[True] / (df['Is_Significant'][df['T_a(°C)'] == temp].value_counts()[True] + df['Is_Significant'][df['T_a(°C)'] == temp].value_counts()[False]) * 100:.2f}%")# 显著数据点的比例
    for season in ['Spring', 'Summer', 'Autumn', 'Winter']:
        N_season = df['N_inp_net(#/L)'][df['T_a(°C)'] == temp][df['season'] == season]
        if len(N_season) == 0:
            continue
        print(f'season: {season}')
        print("Median:", N_season.median())
        print("Mean:", N_season.mean())
        #print("Geometric Mean:", N_season.prod() ** (1 / len(N_season)))
        print(f"Significant data points: {df['Is_Significant'][df['T_a(°C)'] == temp][df['season'] == season].value_counts()[True]}")# 显著数据点的个数
        print(f"{df['Is_Significant'][df['T_a(°C)'] == temp][df['season'] == season].value_counts()[True] / (df['Is_Significant'][df['T_a(°C)'] == temp][df['season'] == season].value_counts()[True] + df['Is_Significant'][df['T_a(°C)'] == temp][df['season'] == season].value_counts()[False]) * 100:.2f}%")# 显著数据点的比例
    print("\n")

In [ ]:
import pandas as pd

# 读取数据
df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.1.csv")

# 创建一个空列表，用于存储每一行的数据
results_list = []

for temp in [-15, -20, -25, -30, -35]:
    # 筛选当前温度的数据
    mask_temp = df['T_a(°C)'] == temp
    df_temp = df[mask_temp]
    
    if len(df_temp) == 0:
        continue
        
    # 计算整体统计量
    med = df_temp['N_inp_net(#/L)'].median()
    mean = df_temp['N_inp_net(#/L)'].mean()
    
    # 计算显著点个数和比例 (Is_Significant为布尔值，直接sum相当于统计True的数量)
    sig_count = df_temp['Is_Significant'].sum()
    total_count = len(df_temp)
    sig_ratio = (sig_count / total_count * 100) if total_count > 0 else 0
    
    # 把整体结果保存到列表中（指定 Season 为 'Overall' 表示该温度整体）
    results_list.append({
        'Temperature(°C)': temp,
        'Season': 'Overall',
        'Median': round(med, 2),
        'Mean': round(mean, 2),
        'Significant_Count': int(sig_count),
        'Significant_Ratio(%)': round(sig_ratio, 2)
    })
    
    # 按照季节进一步筛选
    for season in ['Spring', 'Summer', 'Autumn', 'Winter']:
        mask_season = mask_temp & (df['season'] == season)
        df_season = df[mask_season]
        
        if len(df_season) == 0:
            continue
            
        med_s = df_season['N_inp_net(#/L)'].median()
        mean_s = df_season['N_inp_net(#/L)'].mean()
        
        sig_count_s = df_season['Is_Significant'].sum()
        total_count_s = len(df_season)
        sig_ratio_s = (sig_count_s / total_count_s * 100) if total_count_s > 0 else 0
        
        # 把各季节结果保存到列表中
        results_list.append({
            'Temperature(°C)': temp,
            'Season': season,
            'Median': round(med_s, 2),
            'Mean': round(mean_s, 2),
            'Significant_Count': int(sig_count_s),
            'Significant_Ratio(%)': round(sig_ratio_s, 2)
        })

# 将列表转换为 DataFrame
results_df = pd.DataFrame(results_list)

# 打印预览结果
print(results_df)

# ================= 导出文件 =================

# 1. 保存为 CSV 文件
# encoding='utf-8-sig' 确保用 Excel 打开 csv 时中文或特殊符号(如°C)不会乱码
csv_output_path = r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP_Statistics.csv"
results_df.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
print(f"\n成功保存为 CSV 文件: {csv_output_path}")

# 2. 保存为 Excel 文件 
# (注意：需要安装 openpyxl 库，如果没有安装可以运行 pip install openpyxl)
excel_output_path = r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP_Statistics.xlsx"
results_df.to_excel(excel_output_path, index=False)
print(f"成功保存为 Excel 文件: {excel_output_path}")

## 草稿

In [ ]:
# 给时间序列添加四季背景色
season_colors = {
    'Spring': 'lightskyblue',
    'Summer': 'lightgreen',
    'Autumn': 'moccasin',
    'Winter': 'lavender'
}
season_series = df_selected['season']
#print(season_series)
season_blocks = season_series.ne(season_series.shift()).cumsum()
#print(season_blocks)
#added_labels = set()
#for _, block in season_series.groupby(season_blocks):
#    start, end = block.index.min(), block.index.max()
#    season_name = block.iloc[0]
#    color = season_colors.get(season_name, 'lightgray')
#    #label = season_name if season_name not in added_labels else None
#    ax.axvspan(start, end, color=color, alpha=0.15)
#    #added_labels.add(season_name)